## 网格伪影 Gridding Artifacts

语义分割模型（如 FCN、U-Net、DRN 等）中，网格伪影的根源主要有 3 点：

- 空洞卷积的感受野不连续：
  - 空洞卷积（Dilated Conv）通过在卷积核中插入「空洞」扩大感受野，但如果多个空洞卷积层叠加，空洞率设计不当，会导致感受野出现「不重叠的网格区域」—— 特征图中每个位置的信息只来自输入的离散网格点，最终输出呈现规则网格纹理。
- 编码器下采样的「像素对齐偏差」：
  - 编码器通过池化或步幅卷积下采样（如 ResNet 的 stride=2），会导致特征图的空间位置与原始输入图像出现「网格状错位」；解码器上采样时（如转置卷积、双线性插值），这种错位会被放大，形成可见的网格边界。
- 多尺度特征融合的「通道 / 空间不匹配」：
  - 不同层级的特征图（如浅层高分辨率特征、深层低分辨率特征）融合时，若未处理好通道权重或空间对齐，会导致特征响应呈现网格状分布（比如深层特征的网格模式覆盖浅层细节）。
简单说：网格伪影是模型「空间感知不连续」的外在表现，会导致分割结果中出现无意义的规则线条，降低分割精度（尤其是边界区域）。

解决方法 ==待补充==

## Conv1x1和线性层的等价性

- `Conv1x1` 卷积层示例代码：

In [27]:
import torch 
import torch.nn as nn
# seed
torch.manual_seed(0)

input_tensor = torch.randn(1, 3, 32, 32)  # 示例输入张量，形状为 (batch_size, channels, height, width)
in_planes, out_planes = 3, 10
conv1x1 = nn.Conv2d(in_planes, out_planes, kernel_size=1, stride=1, bias=False)
output_tensor = conv1x1(input_tensor)
print("Input shape:", input_tensor.shape)
# print(input_tensor[0])
print("Conv1x1 weights shape:", conv1x1.weight.shape)  # NT: [10, 3, 1, 1], 10个3x1x1的卷积核
# print(conv1x1.weight[0])
print("Output shape after Conv1x1:", output_tensor.shape)  # NT: [1, 10, 32, 32]: 每个卷积核和输入形成一个32x32的特征图,10个特征图拼接
# print(output_tensor[0][0][0][0])
# print(-1.1258 * 0.5390 + (-1.0841) * 0.4382 + 0.6657 * -0.5052 == output_tensor[0][0][0][0])
print(input_tensor[0][0][0][0] * conv1x1.weight[0][0][0][0] +  # 第一个通道的(0,0)位置 与 第一个卷积核的权重相乘
      input_tensor[0][1][0][0] * conv1x1.weight[0][1][0][0] +
      input_tensor[0][2][0][0] * conv1x1.weight[0][2][0][0] 
      == output_tensor[0][0][0][0])

Input shape: torch.Size([1, 3, 32, 32])
Conv1x1 weights shape: torch.Size([10, 3, 1, 1])
Output shape after Conv1x1: torch.Size([1, 10, 32, 32])
tensor(True)


- 线性层示例代码：
- 想应用线性层实现一样的效果，必须把原本第1维度的`C`通道数放到最后一维度上，并把其他维度展平：
```py
input_reshaped = input.permute(0, 2, 3, 1).view(-1, 3)  # (N, H, W, C) -> (N*H*W, C)
```

In [ ]:
import torch
import torch.nn as nn
torch.manual_seed(0)

input_tensor = torch.randn(1, 3, 32, 32)
linear_layer = nn.Linear(3, 10, bias=False)
# 将线性层的权重调整为与 Conv1x1 相同的形状
linear_layer.weight.data = linear_layer.weight.data.view(10, 3)
# print("Linear layer weights shape:", linear_layer.weight.shape)  # NT: [10, 3]
# print(linear_layer.weight[0])
# 调整输入张量形状以适应线性层
input_reshaped = input_tensor.permute(0, 2, 3, 1).contiguous().view(-1, 3)  # NT: [1*32*32, 3]
output_tensor1 = linear_layer(input_reshaped) # NT: [1*32*32, 10]
output_tensor1 = output_tensor1.view(1, 32, 32, 10).permute(0, 3, 1, 2)  # NT: [1, 10, 32, 32]
print("Output shape after Linear layer:", output_tensor1.shape)
print("Outputs are equal:", torch.allclose(output_tensor, output_tensor1))  # 验证输出是否相等

torch.Size([1024, 10])
Output shape after Linear layer: torch.Size([1, 10, 32, 32])
Outputs are equal: True


## 残差连接

ResNet残差连接的核心代码:
```python
def forward(self, x: Tensor) -> Tensor:
    identity = x  # SOL 保存输入作为残差连接的基础
    # NT: 第一个卷积层
    out = self.conv1(x)
    out = self.bn1(out)
    out = self.relu(out)
    # NT: 第二个卷积层
    out = self.conv2(out)
    out = self.bn2(out)
    out = self.relu(out)
    # NT: 第三个卷积层
    out = self.conv3(out)
    out = self.bn3(out)
    # NT: 下采样层（如果存在）
    if self.downsample is not None:
        identity = self.downsample(x)  # SOL: 如果需要，调整输入维度以匹配输出
    # SOL: 残差连接
    out += identity
    out = self.relu(out)  # SOL: 最终激活函数在残差连接后应用
    return out
```

- 残差连接:
  1. 缓解梯度消失：让梯度直接回传
     - $y=F(x)$, 导数$d\text{loss}/dx = d\text{loss}/dy \times dF(x)/dx$
     - $y=x + F(x)$, 导数$d\text{loss}/dx = d\text{loss}/dy \times (1 + dF(x)/dx)$
     - 即使$dF(x)/dx$趋于0, 仍然保证$d\text{loss}/dy$这部分
     - 让100层网络的训练像10层网络一样稳定
     - 底层参数也能获得有效的梯度更新
  2. 保持信息流动：保留原始输入信息
     - 每层输出 = 原始输入 + 学习到的变化, output = input + learned_change
     - 恒等映射捷径：原始输入 x 直接传递到输出
     - 选择性学习：网络只需要学习相对于输入的"残差" F(x)
     - 信息无损传递：重要特征不会在深层中丢失
  3. 训练稳定性：使深层网络更容易训练
     - 深层网络容易出现的问题:
       - Loss震荡剧烈
       - 需要精心调参和学习率策略
       - 对初始化敏感
     - 训练过程更加稳定和平滑
     - 对超参数的敏感性降低
     - 可以使用更大的学习率加速收敛